In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from prophet import Prophet
import warnings 

warnings.filterwarnings('ignore')

df = pd.read_csv('retail_store_inventory.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

df = pd.get_dummies(df, columns = ['Category', 'Region', 'Seasonality', 'Weather Condition', 'Store ID', 'Product ID'])

q1 = df['Demand Forecast'].quantile(0.25)
q3 = df['Demand Forecast'].quantile(0.75)
iqr = q3 - q1

upper = q3 + 1.5 *iqr
lower = q1 - 1.5 *iqr

df = df[(df['Demand Forecast'] <= upper) & (df['Demand Forecast'] >= lower)]

df = df.rename(columns = {'Date' : 'ds', 'Demand Forecast' : 'y'})

dp = df[['ds', 'y']].copy()

train_xg = df[(df['ds'] >= pd.Timestamp('2022-01-01')) & (df['ds'] <= pd.Timestamp('2023-06-01'))]
test_xg = df[(df['ds'] >= pd.Timestamp('2023-06-02')) & (df['ds'] <= pd.Timestamp('2024-01-01'))]

train = dp[(dp['ds'] >= pd.Timestamp('2022-01-01')) & (dp['ds'] <= pd.Timestamp('2023-06-01'))]
test = dp[(dp['ds'] >= pd.Timestamp('2023-06-02')) & (dp['ds'] <= pd.Timestamp('2024-01-01'))]

model = Prophet(changepoint_prior_scale= 0.001, seasonality_prior_scale= 1, daily_seasonality= True, weekly_seasonality= True,
                yearly_seasonality= True, holidays_prior_scale= 5)
model.fit(train)

predict_train = model.predict(train)
predict_test = model.predict(test)

predict_train = predict_train['yhat'].values
predict_test = predict_test['yhat'].values

y_train_prop = train['y'].values
y_test_prop = test['y'].values

mae_train = mean_absolute_error(y_train_prop, predict_train)
print ("TRAIN ERROR:%.2f"%mae_train)
mae_test = mean_absolute_error(y_test_prop, predict_test)
print ("TEST ERROR:%.2f"%mae_test)

train_xg['Predictions'] = predict_train
train_xg['Residuals'] = train_xg['y'] - train_xg['Predictions']

test_xg['Predictions'] = predict_test
test_xg['Residuals'] = test_xg['y'] - test_xg['Predictions']

train_xg = train_xg.set_index('ds')
test_xg = test_xg.set_index('ds')

def time(df):
    df['Month'] = df.index.month
    df['Year'] = df.index.year
    df['Day'] = df.index.day 
    df['Dayofweek'] = df.index.dayofweek
    df['Dayofyear'] = df.index.dayofyear
    return df

def features(df):
    df['Lag 1'] = df['Residuals'].shift(1)
    df['Lag 7'] = df['Residuals'].shift(7)
    df['Lag 30'] = df['Residuals'].shift(-30)
    df['MA 7'] = df['Residuals'].shift(1).rolling(7).mean()
    df['MA 30'] = df['Residuals'].shift(1).rolling(30).mean()
    df['STD 7'] = df['Residuals'].shift(1).rolling(7).std()
    df['STD 30'] = df['Residuals'].shift(1).rolling(30).std()
    return df

train_xg = time(train_xg)
train_xg = features(train_xg)
test_xg = time(test_xg)
test_xg = features(test_xg)

train_xg.dropna(inplace = True)
test_xg.dropna(inplace = True)

x_train_xg = train_xg.drop(['Residuals', 'y'], axis = 1)
y_train_xg = train_xg['Residuals']

x_test_xg = test_xg.drop(['Residuals', 'y'], axis = 1)
y_test_xg = test_xg['Residuals']

xg = XGBRegressor(n_estimators = 1500, learning_rate = 0.005, max_depth = 5)
xg.fit(x_train_xg, y_train_xg)

pred_train = xg.predict(x_train_xg)
pred_test = xg.predict(x_test_xg)

pred_final_train = train_xg['Predictions'].values + pred_train
pred_final_test = test_xg['Predictions'].values + pred_test

y_train_final = train_xg['y'].values
y_test_final = test_xg['y'].values

mae_final_train = mean_absolute_error(y_train_final,
                                      pred_final_train)
print ("TRAIN FINAL ERROR:%.2f"%mae_final_train)

mae_final_test = mean_absolute_error(y_test_final,
                                     pred_final_test)
print ("TEST FINAL ERROR:%.2f"%mae_final_test)

: 

In [ ]:
df['y'].mean()

: 